# Compare thigh, as opposed to to whole body based, segmentation to ground truth

In [ ]:
# libraries
import os
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.colors import ListedColormap
from ipywidgets import interact, fixed
from IPython.display import clear_output
import SimpleITK as sitk

In [ ]:
!pwd

In [ ]:
results_r_gracilis = []
for file in os.listdir(os.path.join("..","MuscleMap_thigh_segs")):
    print(file)
    if file[0] == 'H':
        gt_name = file[:7]+"/SegmentationMasks/combined_gt_stack"+ file[-13] +".mha"
    else:
        gt_name = file[:6]+"/SegmentationMasks/combined_gt_stack"+ file[-13] +".mha"
        
    print(gt_name)
    segment_image = sitk.ReadImage(os.path.join("..","MuscleMap_thigh_segs",file))
    new_segment_array = sitk.GetArrayFromImage(segment_image)
    gt_image= sitk.ReadImage(gt_name)
    segment_image.CopyInformation(gt_image)
    # actually lower
    #results = []
    gt = sitk.Cast(gt_image ==  5, sitk.sitkUInt8)
    pred = sitk.Cast(segment_image == 12, sitk.sitkUInt8)
    
    gt_arr = sitk.GetArrayFromImage(gt)
    pred_arr = sitk.GetArrayFromImage(pred)
    
    print("GT voxels:", np.sum(gt_arr))
    print("Pred voxels:", np.sum(pred_arr))
    dice_filter = sitk.LabelOverlapMeasuresImageFilter()
    dice_filter.Execute(gt, pred)
    
    right_grac_dice_lower = dice_filter.GetDiceCoefficient()
    right_JaccardCoeffi  = dice_filter.GetJaccardCoefficient()
    right_VolumeSimilar = dice_filter.GetVolumeSimilarity()
    right_FalseNegative = dice_filter.GetFalseNegativeError()
    right_FalsePositive  = dice_filter.GetFalsePositiveError()
    #print("Right gracilis lower dice:", right_grac_dice_lower)
    hd_filter = sitk.HausdorffDistanceImageFilter()
    hd_filter.Execute(gt, pred)
    
    right_grac_hd_lower = hd_filter.GetHausdorffDistance()
    #print("Right gracilis lower Hausdorff distance:", right_grac_hd_lower)
    results_r_gracilis.append({
            "image": gt_name,
            "pred_label": file,
            "R_gracilis_lower_dice:": right_grac_dice_lower,
            "R_gracilis_lower_Hausdorff:": right_grac_hd_lower,
            "R_gracilis_jaccard":right_JaccardCoeffi,
            "R_gracilis_volume_similarity":right_VolumeSimilar,
            "R_gracilis_falseNegative":right_FalseNegative,
            "R_gracilis_falsePostivie":right_FalsePositive,
        })
df_r_gracilis = pd.DataFrame(results_r_gracilis)
df_r_gracilis#.to_csv('../MuscleMap_results_WB/df_r_gracilis.csv')

In [ ]:
!pwd

In [ ]:
old = pd.read_csv('../MuscleMap_results_WB/df_r_gracilis.csv')
old